In [0]:
from pyspark.sql import functions as F

source_path = (
    "/Volumes/ask_sarthak/bronze/"
    "finance_source_files/journal_lines"
)

finance_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(source_path)
)

print("Source records:", finance_df.count())

display(finance_df.limit(10))

5.3 Add ingestion metadata

In [0]:
bronze_df = (
    finance_df
    .withColumn(
        "_ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_source_system",
        F.lit("SYNTHETIC_SAP")
    )
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )
)

display(bronze_df.limit(5))


5.4 Write to a managed Delta table

In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "ask_sarthak.bronze.gl_transactions_raw"
    )
)

print("Bronze Delta table created!")


5.5 Validate the Bronze table

In [0]:
bronze_table = spark.table(
    "ask_sarthak.bronze.gl_transactions_raw"
)

print("Bronze record count:", bronze_table.count())

display(bronze_table.limit(10))


5.6 Check debit and credit totals

In [0]:
display(
    spark.sql("""
        SELECT
            debit_credit,
            COUNT(*) AS record_count,
            SUM(amount) AS total_amount
        FROM ask_sarthak.bronze.gl_transactions_raw
        GROUP BY debit_credit
    """)
)

In [0]:
%sql

SHOW TABLES IN ask_sarthak.bronze;